# 03 — Estratégia de negócio e interpretação

Este notebook **não refaz o GridSearchCV**. Ele lê os artefatos produzidos pelo notebook 02, mantendo a etapa de análise posterior rápida e evitando treinamento repetido.

As métricas de campanha foram calculadas com previsões out-of-fold do conjunto de treino.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.modeling import load_generated_results

result = load_generated_results(ROOT / "reports" / "generated")

## Campanha por capacidade

In [ ]:
display(result["top_k"].round(4))

O `lift_at_k` compara a taxa de churn encontrada no grupo priorizado com a prevalência observada no treino. Valores acima de 1 indicam enriquecimento em relação a uma seleção não direcionada.

## Threshold

In [ ]:
curve = result["thresholds"].copy()
display(curve[["threshold", "contact_rate", "precision", "recall", "f1"]].round(4))

plt.figure(figsize=(8, 5))
plt.plot(curve["threshold"], curve["precision"], label="Precision")
plt.plot(curve["threshold"], curve["recall"], label="Recall")
plt.plot(curve["threshold"], curve["f1"], label="F1")
plt.xlabel("Threshold")
plt.ylabel("Métrica")
plt.legend()
plt.tight_layout()
plt.show()

Sem custo de contato, custo do incentivo, valor do cliente e probabilidade de sucesso da retenção, não há base para chamar um threshold de **ótimo econômico**. O gráfico serve para explicitar o trade-off entre alcance e precisão.

## Interpretação do modelo

In [ ]:
if result["coefficients"] is not None:
    display(
        result["coefficients"]
        [["feature", "coefficient", "odds_ratio"]]
        .head(20)
        .round(4)
    )
else:
    print("Não há tabela de coeficientes salva para o modelo selecionado.")

Os coeficientes descrevem associações no modelo multivariado e **não demonstram causalidade**. Como o One-Hot Encoder mantém todas as categorias, os odds ratios das dummies não devem ser apresentados como comparação simples contra uma categoria omitida. A leitura principal deve considerar direção, magnitude e coerência com a EDA.